# Entraînement des Modèles de Classification Multi-Label

In [1]:
# Importations
import sys
import os
sys.path.append('..')

from notebooks import DataLoaderExploration,Config,DataPreprocessor,FeatureEngineer,ClassifierChainsModel,BinaryRelevanceModel,\
DataVisualizer,ModelEvaluator,MultiLabelMetrics
# from src.utils.metrics import MultiLabelEvaluator
# from src.utils.visualization import ResultsVisualizer

import pyspark.sql.functions as F
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json


In [2]:
# Initialisation
print(" Initialisation du notebook d'entraînement...")
def _create_spark_session():
        """Créer une session Spark optimisée"""
        return SparkSession.builder \
            .appName("models_train") \
            .config("spark.executor.memory", "4g") \
            .config("spark.driver.memory", "2g") \
            .getOrCreate()
            # .master(master) \
            # .config('spark.driver.memory', memory) \
            # .config('spark.driver.executor.memory', executor_memory) \
            # .config('spark.sql.adaptive.enabled', 'true') \
            # .config('spark.sql.adaptive.coalescePartitions.enabled', 'true') \
    
            
spark=_create_spark_session()
print(" Session Spark cree...")
# Charger la configuration
config = Config()
data_config = config.get_data_config()

# Initialiser les composants
loader = DataLoaderExploration(spark, Config)
preprocessor = DataPreprocessor(loader.spark)
evaluator_ = MultiLabelMetrics()
evaluator=ModelEvaluator(loader.spark)
visualizer = DataVisualizer()

print(" Composants initialisés")

 Initialisation du notebook d'entraînement...
 Session Spark cree...
 Composants initialisés


26/01/08 08:14:35 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


##  Chargement des données 

In [3]:
# 2. NETTOYER LE CACHE
spark.catalog.clearCache()
print(" Cache nettoyé")
print("\n CHARGEMENT DES DONNÉES")
# 3. CHARGER LES DONNÉES
df = loader.load_json_data("../data/processed/processed_arxiv_data.json")

loader.explore_data(df)


 Cache nettoyé

 CHARGEMENT DES DONNÉES
Chargement des données depuis ../data/processed/processed_arxiv_data.json...


 50,382 articles chargés

 =Exploration basique des données ===


Nombre de lignes: 50382
Nombre de colonnes: 11

Schéma:
root
 |-- abstract: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- category_list: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- clean_abstract: string (nullable = true)
 |-- clean_title: string (nullable = true)
 |-- combined_text: string (nullable = true)
 |-- domain: string (nullable = true)
 |-- id: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- num_categories: long (nullable = true)
 |-- title: string (nullable = true)


Premières 5 lignes:
+--------------------------------------------------+---------------------------------+------------------------------------+--------------------------------------------------+--------------------------------------------------+--------------------------------------------------+--------+---------+------------------+--------------+--------------------------------------------------+
|                             

## FEATURE ENGINEERING

In [4]:
print("\n FEATURE ENGINEERING")

# Créer l'objet FeatureEngineer
feature_engineer = FeatureEngineer(spark, vocab_size=5000)

# Pipeline complet (avec conversion intégrée)
df_features = feature_engineer.run_full_feature_engineering(
    df, 
    text_col="combined_text"
)

print("\n Feature engineering terminé")



 FEATURE ENGINEERING
 Démarrage du feature engineering
 Création du pipeline de traitement de texte


 Création des 2-grammes


 Calcul des statistiques textuelles
Création des features de métadonnées
⚙️ Combinaison des features
 Features à combiner: ['tfidf_features', 'text_length', 'word_count', 'avg_word_length', 'unique_ratio']
Type de vecteur correct (VectorUDT)
 Vecteur de features créé (5 dimensions)
 Feature engineering terminé

 Feature engineering terminé


## VÉRIFICATION DES VECTEUR

In [5]:
print("\n VÉRIFICATION DES VECTEURS")


# 1. Vérifier le schéma
print("\n1️ Schéma de la colonne 'features':")
df_features.select("features").printSchema()

# 2. Vérifier les dimensions
print("\n2️ Vérification des dimensions:")
samples = df_features.select("features").take(10)
sizes = []

for i, row in enumerate(samples):
    vec = row['features']
    if hasattr(vec, 'size'):
        sizes.append(vec.size)
        if i < 3:
            print(f"   Vecteur {i+1}: type={type(vec).__name__}, size={vec.size}")
    else:
        print(f"    Vecteur {i+1}: type incorrect = {type(vec)}")

# Vérifier l'unicité des tailles
unique_sizes = set(sizes)
if len(unique_sizes) == 1:
    print(f"\n SUCCÈS : Tous les vecteurs ont la même dimension = {unique_sizes.pop()}")
else:
    print(f"\nERREUR : Dimensions différentes trouvées = {unique_sizes}")
    raise ValueError("Dimensions de vecteurs incohérentes!")



 VÉRIFICATION DES VECTEURS

1️ Schéma de la colonne 'features':
root
 |-- features: vector (nullable = true)


2️ Vérification des dimensions:
   Vecteur 1: type=SparseVector, size=5004
   Vecteur 2: type=SparseVector, size=5004
   Vecteur 3: type=SparseVector, size=5004

 SUCCÈS : Tous les vecteurs ont la même dimension = 5004


## PRÉPARATION DES LABEL

In [6]:
print("\n PRÉPARATION DES LABELS")

# Créer les modèles
br_model = BinaryRelevanceModel(spark)
cc_model = ClassifierChainsModel(spark)

# Préparer les labels (réduire à 15 pour gagner du temps)
df_features = br_model.prepare_labels(df_features, top_n=15)

print(f"\n {len(br_model.label_columns)} labels créés")
print(f" Premiers labels: {br_model.label_columns[:3]}")




 PRÉPARATION DES LABELS


 15 catégories sélectionnées

 15 labels créés
 Premiers labels: ['label_cs_LG', 'label_hep_ph', 'label_cs_CV']


In [7]:
##SPLIT TRAIN/TEST

In [8]:
print("\n SPLIT TRAIN/TEST")
# Split 80/20
train_df, test_df = df_features.randomSplit([0.8, 0.2], seed=42)

train_count = train_df.count()
test_count = test_df.count()

print(f" Train: {train_count:,} articles ({train_count/(train_count+test_count)*100:.1f}%)")
print(f" Test:  {test_count:,} articles ({test_count/(train_count+test_count)*100:.1f}%)")

# Vérifier les dimensions dans les deux datasets
print("\n🔍 Vérification des dimensions:")
train_sample = train_df.select("features").take(5)
test_sample = test_df.select("features").take(5)

train_sizes = {v['features'].size for v in train_sample}
test_sizes = {v['features'].size for v in test_sample}

print(f"   Train sizes: {train_sizes}")
print(f"   Test sizes: {test_sizes}")

if len(train_sizes) == 1 and len(test_sizes) == 1 and train_sizes == test_sizes:
    print(f"    Dimensions cohérentes: {train_sizes.pop()}")
else:
    print(f"    Dimensions incohérentes!")
    raise ValueError("Problème de dimensions!")



 SPLIT TRAIN/TEST


26/01/08 08:15:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

 Train: 40,509 articles (80.4%)
 Test:  9,873 articles (19.6%)

🔍 Vérification des dimensions:


[Stage 23:>                                                         (0 + 1) / 1]

   Train sizes: {5004}
   Test sizes: {5004}
    Dimensions cohérentes: 5004


##  ENTRAÎNEMENT BINARY RELEVANCE

In [9]:
print(" MODÈLE 1: BINARY RELEVANCE")


import time
start_time = time.time()

# Entraîner (SANS aucune conversion supplémentaire)
br_model.train(train_df, features_col="features")

elapsed = time.time() - start_time
print(f"\n Temps d'entraînement: {elapsed/60:.2f} minutes")


 MODÈLE 1: BINARY RELEVANCE
Entraînement Binary Relevance
  Entraînement pour label_cs_LG...


    AUC: 0.9470
  Entraînement pour label_hep_ph...


    AUC: 0.9587
  Entraînement pour label_cs_CV...


    AUC: 0.9812
  Entraînement pour label_hep_th...


    AUC: 0.9477
  Entraînement pour label_quant_ph...


...

    AUC: 0.9536
  Entraînement pour label_math_MP...


    AUC: 0.8515
  Entraînement pour label_math_ph...


    AUC: 0.8515
  Entraînement pour label_cond_mat_stat_mech...


    AUC: 0.8986
  Entraînement pour label_cond_mat_str_el...


    AUC: 0.9593

 Résultats d'entraînement:
  label_cs_LG: AUC = 0.9470
  label_hep_ph: AUC = 0.9587
  label_cs_CV: AUC = 0.9812
  label_hep_th: AUC = 0.9477
  label_quant_ph: AUC = 0.9570
  label_cs_AI: AUC = 0.9251
  label_gr_qc: AUC = 0.9626
  label_astro_ph: AUC = 0.9156
  label_cond_mat_mtrl_sci: AUC = 0.9482
  label_cs_CL: AUC = 0.9761
  label_cond_mat_mes_hall: AUC = 0.9536
  label_math_MP: AUC = 0.8515
  label_math_ph: AUC = 0.8515
  label_cond_mat_stat_mech: AUC = 0.8986
  label_cond_mat_str_el: AUC = 0.9593

 AUC moyenne: 0.9356

 Temps d'entraînement: 18.21 minutes


## PRÉDICTIONS BINARY RELEVANCE

In [10]:
print("\n PRÉDICTIONS - BINARY RELEVANCE")


# Prédire
predictions_br = br_model.predict(test_df, features_col="features")

print(" Prédictions terminées")

# Afficher un exemple
predictions_br.select(
    [col for col in predictions_br.columns if col.endswith('_pred')][:5]
).show(5)



 PRÉDICTIONS - BINARY RELEVANCE
🔮 Prédictions Binary Relevance
  Prédiction pour label_cs_LG...
  Prédiction pour label_hep_ph...
  Prédiction pour label_cs_CV...
  Prédiction pour label_hep_th...
  Prédiction pour label_quant_ph...
  Prédiction pour label_cs_AI...
  Prédiction pour label_gr_qc...
  Prédiction pour label_astro_ph...
  Prédiction pour label_cond_mat_mtrl_sci...
  Prédiction pour label_cs_CL...
  Prédiction pour label_cond_mat_mes_hall...
  Prédiction pour label_math_MP...
  Prédiction pour label_math_ph...
  Prédiction pour label_cond_mat_stat_mech...
  Prédiction pour label_cond_mat_str_el...

✅ Prédictions générées pour 15 labels
 Prédictions terminées


[Stage 1617:>                                                       (0 + 1) / 1]

+----------------+-----------------+----------------+-----------------+-------------------+
|label_cs_LG_pred|label_hep_ph_pred|label_cs_CV_pred|label_hep_th_pred|label_quant_ph_pred|
+----------------+-----------------+----------------+-----------------+-------------------+
|             0.0|              0.0|             0.0|              0.0|                0.0|
|             0.0|              0.0|             0.0|              0.0|                0.0|
|             0.0|              0.0|             1.0|              0.0|                0.0|
|             0.0|              0.0|             0.0|              1.0|                0.0|
|             0.0|              0.0|             0.0|              0.0|                0.0|
+----------------+-----------------+----------------+-----------------+-------------------+
only showing top 5 rows



## ENTRAÎNEMENT CLASSIFIER CHAINS

In [ ]:
print(" MODÈLE 2: CLASSIFIER CHAINS")

start_time = time.time()

# Entraîner
cc_model.train(
    train_df, 
    label_columns=br_model.label_columns,
    features_col="features"
)

elapsed = time.time() - start_time
print(f"\n Temps d'entraînement: {elapsed/60:.2f} minutes")

 MODÈLE 2: CLASSIFIER CHAINS
🔗 Entraînement Classifier Chains


[Stage 1621:>                                                       (0 + 8) / 8]

In [ ]:
## PRÉDICTIONS CLASSIFIER CHAINS

In [ ]:
print("\n PRÉDICTIONS - CLASSIFIER CHAINS")

# Prédire
predictions_cc = cc_model.predict(test_df, features_col="features")

print(" Prédictions terminées")

# Afficher un exemple
predictions_cc.select(
    [col for col in predictions_cc.columns if col.endswith('_pred')][:5]
).show(5)


##  ÉVALUATION

In [ ]:
print(" ÉVALUATION DES MODÈLES")

from src.utils.metrics import MultiLabelMetrics
import numpy as np

# Préparer les données pour l'évaluation
def prepare_for_evaluation(predictions_df, label_columns):
    """Convertit les prédictions Spark en arrays NumPy"""
    
    # Colonnes de prédiction
    pred_cols = [f"{label}_pred" for label in label_columns]
    
    # Collecter les données
    data = predictions_df.select(label_columns + pred_cols).collect()
    
    # Convertir en arrays
    y_true = np.array([[row[label] for label in label_columns] for row in data])
    y_pred = np.array([[row[pred_col] for pred_col in pred_cols] for row in data])
    
    return y_true, y_pred

# Évaluer Binary Relevance
print("\n BINARY RELEVANCE")
y_true_br, y_pred_br = prepare_for_evaluation(predictions_br, br_model.label_columns)

hamming_br = MultiLabelMetrics.calculate_hamming_loss(y_true_br, y_pred_br)
subset_br = MultiLabelMetrics.calculate_subset_accuracy(y_true_br, y_pred_br)

print(f"   Hamming Loss: {hamming_br:.4f}")
print(f"   Subset Accuracy: {subset_br:.4f}")

# Évaluer Classifier Chains
print("\n2️ CLASSIFIER CHAINS")
y_true_cc, y_pred_cc = prepare_for_evaluation(predictions_cc, cc_model.label_order)

hamming_cc = MultiLabelMetrics.calculate_hamming_loss(y_true_cc, y_pred_cc)
subset_cc = MultiLabelMetrics.calculate_subset_accuracy(y_true_cc, y_pred_cc)

print(f"   Hamming Loss: {hamming_cc:.4f}")
print(f"   Subset Accuracy: {subset_cc:.4f}")

# Comparaison
print("\n" + "="*60)
print(" COMPARAISON")
print("="*60)
print(f"Binary Relevance   - Hamming Loss: {hamming_br:.4f} | Subset Acc: {subset_br:.4f}")
print(f"Classifier Chains  - Hamming Loss: {hamming_cc:.4f} | Subset Acc: {subset_cc:.4f}")

if hamming_br < hamming_cc:
    print("\n Meilleur modèle: BINARY RELEVANCE")
else:
    print("\n Meilleur modèle: CLASSIFIER CHAINS")


##  Approche 1: Binary Relevance

## Nettoyage

In [ ]:
# Nettoyer la mémoire
print("\n Nettoyage de la mémoire...")

train_df.unpersist()
test_df.unpersist()

# Arrêter la session Spark
loader.spark.stop()

print("\n" + "="*60)
print(" NOTEBOOK TERMINÉ AVEC SUCCÈS!")
